In [1]:
import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import pandas as pd
import time
from datetime import datetime, timezone
from sqlalchemy import create_engine
from pprint import pprint

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r
%store

Stored variables and their in-db values:
alarm_names                                -> ['cls-f1-141e5d13', 'cls-input-brightness-drift-14
baseline_prefix                            -> 'cat-landmarks-project/monitoring/f0d83606-c3c6-4b
bucket                                     -> 'sagemaker-us-east-1-298748835671'
cls_drift_schedule                         -> 'cls-drift-hourly-141e5d13'
cls_endpoint_name                          -> 'cls-f0d83606-c3c6-4b88-a55b-63efac0bccfc'
cls_model_data_url                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
cls_model_package_arn                      -> 'arn:aws:sagemaker:us-east-1:298748835671:model-pa
cls_quality_schedule                       -> 'cls-quality-hourly-141e5d13'
data_capture_prefix                        -> 'cat-landmarks-project/monitoring/f0d83606-c3c6-4b
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                    

In [3]:
# Reassigning store variables for IDE convenience
bucket = bucket
database_name = database_name
project_prefix = project_prefix
landmarks_table = landmarks_table
s3_staging_dir = s3_staging_dir
s3_raw_cats_prefix = s3_raw_cats_prefix

In [4]:
s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)


# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

s3_combined_location = (
    f"s3://{bucket}/cat-landmarks-project/processed/combined/image_landmarks_features/"
)

In [5]:
statement_landmarks_table = f"""
SELECT *
FROM {database_name}.{landmarks_table} 
"""

landmarks_df = pd.read_sql(statement_landmarks_table, engine)
landmarks_df.head(2)

,remote_path,local_path,left_eye,right_eye,mouth,left_ear_1,left_ear_2,left_ear_3,right_ear_1,right_ear_2,...,right_ear_1_x,right_ear_1_y,right_ear_2_x,right_ear_2_y,right_ear_3_x,right_ear_3_y,file_size,width,height,label
0,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/cats/CAT_00/00000001_000.jpg,"[175, 160]","[239, 162]","[199, 199]","[149, 121]","[137, 78]","[166, 93]","[281, 101]","[312, 96]",...,281,101,312,96,296,133,62243,375,500,1
1,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/cats/CAT_00/00000001_005.jpg,"[96, 96]","[153, 127]","[103, 146]","[48, 20]","[78, -37]","[127, 5]","[210, 52]","[260, 63]",...,210,52,260,63,229,121,53666,500,375,1


## Feature Store

In [6]:
# Initiate a Feature Store client
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

# Initiate a Feature Store Session
feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

## Ingest Data into FeatureStore

In [7]:
# Create feature definitions
def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")

record_identifier_feature_name = "remote_path"
event_time_feature_name = "event_time"
current_time_sec = datetime.now(timezone.utc).isoformat()
landmarks_df[event_time_feature_name] = current_time_sec

# Data validation
if landmarks_df[record_identifier_feature_name].isna().any():
    raise ValueError("remote_path contains nulls. Feature Store record identifier cannot be null.")

if not landmarks_df[record_identifier_feature_name].is_unique:
    raise ValueError("remote_path must be unique. You have duplicates (would overwrite records).")

# Cast object dtype to str
cast_object_to_string(landmarks_df)

landmarks_feature_group_name = "landmarks-feature-group-" + time.strftime("%d-%H-%M-%S", time.gmtime())

landmarks_feature_group = FeatureGroup(
    name=landmarks_feature_group_name, sagemaker_session=feature_store_session
)

# Load feature definitions (schema inference)
landmarks_feature_group.load_feature_definitions(data_frame=landmarks_df)

[FeatureDefinition(feature_name='remote_path', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='local_path', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='left_eye', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='right_eye', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='mouth', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='left_ear_1', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='left_ear_2', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='left_ear_3', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='right_ear_1', feature_type=<Fe

In [8]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

In [9]:
landmarks_feature_group.create(
    s3_uri=f"s3://{bucket}/{project_prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=landmarks_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup landmarks-feature-group-12-03-00-19 successfully created.


In [10]:
# Ingest data into feature store
landmarks_feature_group.ingest(data_frame=landmarks_df, 
                               max_workers=3, 
                               wait=True)

IngestionManagerPandas(feature_group_name='landmarks-feature-group-12-03-00-19', feature_definitions={'remote_path': {'FeatureName': 'remote_path', 'FeatureType': 'String'}, 'local_path': {'FeatureName': 'local_path', 'FeatureType': 'String'}, 'left_eye': {'FeatureName': 'left_eye', 'FeatureType': 'String'}, 'right_eye': {'FeatureName': 'right_eye', 'FeatureType': 'String'}, 'mouth': {'FeatureName': 'mouth', 'FeatureType': 'String'}, 'left_ear_1': {'FeatureName': 'left_ear_1', 'FeatureType': 'String'}, 'left_ear_2': {'FeatureName': 'left_ear_2', 'FeatureType': 'String'}, 'left_ear_3': {'FeatureName': 'left_ear_3', 'FeatureType': 'String'}, 'right_ear_1': {'FeatureName': 'right_ear_1', 'FeatureType': 'String'}, 'right_ear_2': {'FeatureName': 'right_ear_2', 'FeatureType': 'String'}, 'right_ear_3': {'FeatureName': 'right_ear_3', 'FeatureType': 'String'}, 'version': {'FeatureName': 'version', 'FeatureType': 'String'}, 'split': {'FeatureName': 'split', 'FeatureType': 'String'}, 'left_eye_x'

In [11]:
#  Verifying with feature record
record_identifier_value = landmarks_df["remote_path"].values[0]

fg_name = landmarks_feature_group.name  

response = featurestore_runtime.get_record(
    FeatureGroupName=fg_name,
    RecordIdentifierValueAsString=record_identifier_value,
)

pprint(response["Record"])

[{'FeatureName': 'remote_path',
  'ValueAsString': 's3://sagemaker-us-east-1-298748835671/cat-landmarks-project/data/raw/cats/CAT_00/00000001_000.jpg'},
 {'FeatureName': 'local_path',
  'ValueAsString': 'data/raw/cats/CAT_00/00000001_000.jpg'},
 {'FeatureName': 'left_eye', 'ValueAsString': '[175, 160]'},
 {'FeatureName': 'right_eye', 'ValueAsString': '[239, 162]'},
 {'FeatureName': 'mouth', 'ValueAsString': '[199, 199]'},
 {'FeatureName': 'left_ear_1', 'ValueAsString': '[149, 121]'},
 {'FeatureName': 'left_ear_2', 'ValueAsString': '[137, 78]'},
 {'FeatureName': 'left_ear_3', 'ValueAsString': '[166, 93]'},
 {'FeatureName': 'right_ear_1', 'ValueAsString': '[281, 101]'},
 {'FeatureName': 'right_ear_2', 'ValueAsString': '[312, 96]'},
 {'FeatureName': 'right_ear_3', 'ValueAsString': '[296, 133]'},
 {'FeatureName': 'version', 'ValueAsString': 'raw'},
 {'FeatureName': 'split', 'ValueAsString': 'production'},
 {'FeatureName': 'left_eye_x', 'ValueAsString': '175'},
 {'FeatureName': 'left_eye_y'

In [12]:
fg_status = landmarks_feature_group.describe()
pprint(fg_status)

{'CreationTime': datetime.datetime(2026, 10, 12, 3, 0, 19, 847000, tzinfo=tzlocal()),
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'remote_path', 'FeatureType': 'String'},
                        {'FeatureName': 'local_path', 'FeatureType': 'String'},
                        {'FeatureName': 'left_eye', 'FeatureType': 'String'},
                        {'FeatureName': 'right_eye', 'FeatureType': 'String'},
                        {'FeatureName': 'mouth', 'FeatureType': 'String'},
                        {'FeatureName': 'left_ear_1', 'FeatureType': 'String'},
                        {'FeatureName': 'left_ear_2', 'FeatureType': 'String'},
                        {'FeatureName': 'left_ear_3', 'FeatureType': 'String'},
                        {'FeatureName': 'right_ear_1', 'FeatureType': 'String'},
                        {'FeatureName': 'right_ear_2', 'FeatureType': 'String'},
                        {'FeatureName': 'right_ear_3', 'FeatureType': 'String'},

In [13]:
%store landmarks_feature_group_name

Stored 'landmarks_feature_group_name' (str)
